In [0]:
spark.table("workspace.logistics_project.routes").printSchema()

root
 |-- route_id: string (nullable = true)
 |-- origin_city: string (nullable = true)
 |-- origin_state: string (nullable = true)
 |-- destination_city: string (nullable = true)
 |-- destination_state: string (nullable = true)
 |-- typical_distance_miles: long (nullable = true)
 |-- base_rate_per_mile: double (nullable = true)
 |-- fuel_surcharge_rate: double (nullable = true)
 |-- typical_transit_days: long (nullable = true)



In [0]:
spark.table("workspace.logistics_project.fuel_purchases").printSchema()

root
 |-- fuel_purchase_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- truck_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- purchase_date: timestamp (nullable = true)
 |-- location_city: string (nullable = true)
 |-- location_state: string (nullable = true)
 |-- gallons: double (nullable = true)
 |-- price_per_gallon: double (nullable = true)
 |-- total_cost: double (nullable = true)
 |-- fuel_card_number: string (nullable = true)
 |-- month: date (nullable = true)
 |-- trips_completed: long (nullable = true)
 |-- total_miles: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- average_mpg: double (nullable = true)
 |-- maintenance_events: long (nullable = true)
 |-- maintenance_cost: double (nullable = true)
 |-- downtime_hours: double (nullable = true)
 |-- utilization_rate: double (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_type: stri

In [0]:
#Read Tables
routes = spark.table("workspace.logistics_project.routes")
loads = spark.table("workspace.logistics_project.loads")
trips = spark.table("workspace.logistics_project.trips")
fuel_purchases = spark.table("workspace.logistics_project.fuel_purchases")

In [0]:
#Join Tables
route_profit = (
    routes
    .join(loads, "route_id", "inner")
    .join(trips, "load_id", "inner")
    .join(fuel_purchases.select(
        "trip_id",
        "total_cost"
    ), "trip_id", "left")
)

In [0]:
#Calculate Route Metrics
from pyspark.sql.functions import *

route_metrics = (
    route_profit
    .groupBy(
        "route_id",
        "origin_city",
        "destination_city"
    )
    .agg(
        round(sum("revenue"),2).alias("total_revenue"),

        round(sum("total_cost"),2).alias("fuel_cost"),

        sum("actual_distance_miles").alias("total_miles"),

        countDistinct("trip_id").alias("total_trips")
    )
)

In [0]:
#Calculate Profit
route_metrics = route_metrics.withColumn(
    "profit",
    round(
        col("total_revenue") - col("fuel_cost"),
        2
    )
)

route_metrics = route_metrics.withColumn(
    "profit_per_mile",
    round(
        col("profit") / col("total_miles"),
        2
    )
)

In [0]:
#View Results
display(
    route_metrics.orderBy(
        col("profit").desc()
    )
)

route_id,origin_city,destination_city,total_revenue,fuel_cost,total_miles,total_trips,profit,profit_per_mile
RTE00016,Philadelphia,Seattle,3.292078752E7,1608382.4,13469702,1465,3.131240512E7,2.32
RTE00044,Charlotte,Portland,3.255648555E7,1553387.9,12454842,1410,3.100309765E7,2.49
RTE00029,Seattle,Charlotte,3.252379895E7,1707728.59,13555750,1467,3.081607036E7,2.27
RTE00012,Phoenix,Philadelphia,3.219215051E7,1669546.56,12211837,1507,3.052260395E7,2.5
RTE00046,Columbus,Los Angeles,3.135000959E7,1721908.05,11820335,1534,2.962810154E7,2.51
RTE00048,Columbus,Portland,3.125525706E7,1665771.08,11925326,1525,2.958948598E7,2.48
RTE00042,Charlotte,Seattle,2.995120136E7,1615945.96,12928797,1447,2.83352554E7,2.19
RTE00030,Seattle,Indianapolis,2.561516941E7,1644240.2,10639167,1445,2.397092921E7,2.25
RTE00021,Houston,Portland,2.511433429E7,1643261.26,10573979,1473,2.347107303E7,2.22
RTE00003,Chicago,Los Angeles,2.502533182E7,1668535.89,10117537,1496,2.335679593E7,2.31


In [0]:
route_profitability = spark.table(
    "workspace.logistics_gold.route_profitability"
)

display(route_profitability)

route_id,origin_city,destination_city,total_revenue,fuel_cost,total_miles,total_trips,profit,profit_per_mile
RTE00025,Miami,Seattle,2.491667739E7,1641525.7,15512423,1450,2.327515169E7,1.5
RTE00043,Charlotte,Denver,1.184134367E7,1661183.09,7882067,1500,1.018016058E7,1.29
RTE00029,Seattle,Charlotte,3.252379895E7,1707728.59,13555750,1467,3.081607036E7,2.27
RTE00058,Kansas City,Indianapolis,5507259.01,1588173.91,2507550,1432,3919085.1,1.56
RTE00056,Kansas City,Portland,2.060456785E7,1643981.36,8614737,1471,1.896058649E7,2.2
RTE00037,Las Vegas,New York,1.873596071E7,1608384.69,12650738,1477,1.712757602E7,1.35
RTE00049,Columbus,Minneapolis,9010317.51,1599911.65,3531652,1461,7410405.86,2.1
RTE00008,Dallas,Denver,9651616.28,1642198.95,3795661,1461,8009417.33,2.11
RTE00036,Las Vegas,Los Angeles,3469479.01,1699724.63,1358584,1522,1769754.38,1.3
RTE00057,Kansas City,Charlotte,1.079135867E7,1620741.5,4547687,1469,9170617.17,2.02


In [0]:
route_profitability.printSchema()

root
 |-- route_id: string (nullable = true)
 |-- origin_city: string (nullable = true)
 |-- destination_city: string (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- fuel_cost: double (nullable = true)
 |-- total_miles: long (nullable = true)
 |-- total_trips: long (nullable = true)
 |-- profit: double (nullable = true)
 |-- profit_per_mile: double (nullable = true)



In [0]:
display(
    spark.table("workspace.logistics_gold.route_profitability")
)

route_id,origin_city,destination_city,total_revenue,fuel_cost,total_miles,total_trips,profit,profit_per_mile
RTE00025,Miami,Seattle,2.491667739E7,1641525.7,15512423,1450,2.327515169E7,1.5
RTE00043,Charlotte,Denver,1.184134367E7,1661183.09,7882067,1500,1.018016058E7,1.29
RTE00029,Seattle,Charlotte,3.252379895E7,1707728.59,13555750,1467,3.081607036E7,2.27
RTE00058,Kansas City,Indianapolis,5507259.01,1588173.91,2507550,1432,3919085.1,1.56
RTE00056,Kansas City,Portland,2.060456785E7,1643981.36,8614737,1471,1.896058649E7,2.2
RTE00037,Las Vegas,New York,1.873596071E7,1608384.69,12650738,1477,1.712757602E7,1.35
RTE00049,Columbus,Minneapolis,9010317.51,1599911.65,3531652,1461,7410405.86,2.1
RTE00008,Dallas,Denver,9651616.28,1642198.95,3795661,1461,8009417.33,2.11
RTE00036,Las Vegas,Los Angeles,3469479.01,1699724.63,1358584,1522,1769754.38,1.3
RTE00057,Kansas City,Charlotte,1.079135867E7,1620741.5,4547687,1469,9170617.17,2.02


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
#Create Gold Schema
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.logistics_gold
""")

DataFrame[]

In [0]:
spark.sql("""
SHOW TABLES IN workspace.logistics_gold
""").show(truncate=False)

+--------------+------------------+-----------+
|database      |tableName         |isTemporary|
+--------------+------------------+-----------+
|logistics_gold|driver_performance|false      |
+--------------+------------------+-----------+



In [0]:
route_metrics.write \
    .format("delta") \
    .saveAsTable(
        "workspace.logistics_gold.route_profitability"
    )

In [0]:
spark.sql("""
SHOW TABLES IN workspace.logistics_gold
""").show(truncate=False)

+--------------+-------------------+-----------+
|database      |tableName          |isTemporary|
+--------------+-------------------+-----------+
|logistics_gold|driver_performance |false      |
|logistics_gold|route_profitability|false      |
+--------------+-------------------+-----------+



In [0]:
spark.table("workspace.logistics_project.trucks").printSchema()

root
 |-- truck_id: string (nullable = true)
 |-- unit_number: long (nullable = true)
 |-- make: string (nullable = true)
 |-- model_year: long (nullable = true)
 |-- vin: string (nullable = true)
 |-- acquisition_date: date (nullable = true)
 |-- acquisition_mileage: long (nullable = true)
 |-- fuel_type: string (nullable = true)
 |-- tank_capacity_gallons: long (nullable = true)
 |-- status: string (nullable = true)
 |-- home_terminal: string (nullable = true)



In [0]:
spark.table("workspace.logistics_project.truck_utilization_metrics").printSchema()

root
 |-- truck_id: string (nullable = true)
 |-- month: date (nullable = true)
 |-- trips_completed: long (nullable = true)
 |-- total_miles: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- average_mpg: double (nullable = true)
 |-- maintenance_events: long (nullable = true)
 |-- maintenance_cost: double (nullable = true)
 |-- downtime_hours: double (nullable = true)
 |-- utilization_rate: double (nullable = true)

